# NASA Small UAV Audio Detection

This notebook builds a practical audio detector from the NASA Small UAS Acoustic Data `.mat` files in `/home/mo/dev/python/HDDS2-datasets/small_uav_acoustics/`.

Goal:
- read the MATLAB acoustic recordings,
- extract microphone audio windows,
- derive labels from vehicle type and GPS distance,
- train an app-compatible MFCC classifier,
- export a `.joblib` model that the dashboard can use,
- optionally export `.wav` audio files for scenario testing.

The label strategy is intentionally simple and editable:
- `phantom`, `hex`, and `y6` are treated as multirotor drone vehicles.
- `edge` and `cub` are treated as fixed-wing/non-multirotor examples.
- multirotor windows near the microphone are positive.
- far-away multirotor windows and fixed-wing windows are negative/background-like.

In [1]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from dataclasses import dataclass
from pathlib import Path

Path('/tmp/hdds_numba_cache').mkdir(parents=True, exist_ok=True)
Path('/tmp/hdds_matplotlib').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('NUMBA_CACHE_DIR', '/tmp/hdds_numba_cache')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/hdds_matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/hdds_cache')

import joblib
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.io import loadmat
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path('/home/mo/dev/python/HDDS2/Project v1')
DATASET_ROOT = Path('/home/mo/dev/python/HDDS2-datasets/small_uav_acoustics')
MAT_DIR = DATASET_ROOT / 'data'
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from audio.features import extract_baseline_features_from_signal

assert MAT_DIR.exists(), f'Missing dataset directory: {MAT_DIR}'
print('Project:', PROJECT_ROOT)
print('Dataset:', MAT_DIR)

Project: /home/mo/dev/python/HDDS2/Project v1
Dataset: /home/mo/dev/python/HDDS2-datasets/small_uav_acoustics/data


## Configuration

The most important values are the label thresholds.

- `POSITIVE_DISTANCE_M`: multirotor windows at or below this distance are labeled `drone`.
- `BACKGROUND_DISTANCE_M`: multirotor windows beyond this distance are labeled `no_drone`.
- middle-distance multirotor windows are skipped to avoid ambiguous labels.
- fixed-wing `edge` and `cub` windows are labeled `no_drone` by default.

In [2]:
TARGET_SR = 16_000
WINDOW_S = 3.0
HOP_S = 1.5

POSITIVE_DISTANCE_M = 140.0
BACKGROUND_DISTANCE_M = 260.0
INCLUDE_FIXED_WING_AS_NEGATIVE = True
MAX_WINDOWS_PER_FILE = 80  # keep first runs quick; set None to use all eligible windows
RANDOM_SEED = 7

MULTIROTOR_VEHICLES = {'phantom', 'hex', 'y6'}
FIXED_WING_VEHICLES = {'edge', 'cub'}

MODEL_OUT = PROJECT_ROOT / 'src' / 'audio' / 'models' / 'sound_baseline_mfcc_logreg.joblib'
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
print('Model export path:', MODEL_OUT)

Model export path: /home/mo/dev/python/HDDS2/Project v1/src/audio/models/sound_baseline_mfcc_logreg.joblib


## Inspect Available Flights

File names follow this pattern:

```text
vehicle_mode_flightnumber.mat
```

Examples: `phantom_flyover_120.mat`, `hex_hover_194.mat`.

In [3]:
def parse_mat_name(path: Path) -> dict[str, object]:
    match = re.match(r'(?P<vehicle>[^_]+)_(?P<mode>[^_]+)_(?P<flight>\d+)$', path.stem)
    if not match:
        raise ValueError(f'Unexpected file name: {path.name}')
    return {
        'path': path,
        'file': path.name,
        'vehicle': match.group('vehicle').lower(),
        'mode': match.group('mode').lower(),
        'flight': int(match.group('flight')),
    }

files = sorted(MAT_DIR.glob('*.mat'))
flight_df = pd.DataFrame(parse_mat_name(path) for path in files)
display(flight_df.groupby(['vehicle', 'mode']).size().rename('count').reset_index())
print('Total MAT files:', len(flight_df))

,vehicle,mode,count
0,cub,flyover,3
1,edge,flyover,13
2,hex,flyover,14
3,hex,hover,10
4,phantom,flyover,6
5,phantom,hover,5
6,y6,hover,11


Total MAT files: 62


## MATLAB File Helpers

Each `.mat` file contains:

- `acoustics.incident_pascals`: microphone audio as pressure samples, shape `(samples, microphones)`.
- `acoustics.utc_time`: audio sample times.
- `vehicle_data.rtk_ned_meters` / `gps_ned_meters`: vehicle position relative to the origin microphone.

The NASA example scripts use microphone 1 for `hex` and `cub`, and microphone 3 for `phantom`, `edge`, and `y6`. The helper below follows that convention.

In [4]:
def load_mat_struct(path: Path):
    return loadmat(path, squeeze_me=True, struct_as_record=False)


def selected_mic_index(vehicle: str) -> int:
    # Python indexes are zero-based. NASA MATLAB scripts use channel 1 for AP Hill
    # hex/cub and channel 3 for Va Beach phantom/edge/y6.
    return 0 if vehicle in {'hex', 'cub'} else 2


def get_audio_and_time(path: Path, vehicle: str) -> tuple[np.ndarray, np.ndarray, float]:
    data = load_mat_struct(path)
    acoustics = data['acoustics']
    pressure = np.asarray(acoustics.incident_pascals, dtype=np.float32)
    if pressure.ndim == 1:
        audio = pressure
    else:
        mic_index = min(selected_mic_index(vehicle), pressure.shape[1] - 1)
        audio = pressure[:, mic_index]
    utc_time = np.asarray(acoustics.utc_time, dtype=np.float64)
    source_sr = float(1.0 / np.median(np.diff(utc_time)))
    audio = np.nan_to_num(audio, copy=False)
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 1.0:
        audio = 0.95 * audio / peak
    if int(round(source_sr)) != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=source_sr, target_sr=TARGET_SR)
        duration = (utc_time[-1] - utc_time[0]) if utc_time.size > 1 else len(audio) / TARGET_SR
        utc_time = np.linspace(utc_time[0], utc_time[0] + duration, len(audio), endpoint=False)
        source_sr = TARGET_SR
    return audio.astype(np.float32), utc_time, float(source_sr)


def get_position_track(path: Path) -> tuple[np.ndarray | None, np.ndarray | None, str]:
    data = load_mat_struct(path)
    vehicle_data = data['vehicle_data']
    for time_field, pos_field, source in [
        ('rtk_utc_time', 'rtk_ned_meters', 'rtk'),
        ('gps_utc_time', 'gps_ned_meters', 'gps'),
    ]:
        if not hasattr(vehicle_data, time_field) or not hasattr(vehicle_data, pos_field):
            continue
        t = np.asarray(getattr(vehicle_data, time_field), dtype=np.float64)
        xyz = np.asarray(getattr(vehicle_data, pos_field), dtype=np.float64)
        if t.size >= 2 and xyz.ndim == 2 and xyz.shape[0] == t.size:
            order = np.argsort(t)
            return t[order], xyz[order], source
    return None, None, 'none'


def distance_at_times(path: Path, times_utc: np.ndarray) -> tuple[np.ndarray, str]:
    pos_t, ned, source = get_position_track(path)
    if pos_t is None or ned is None:
        return np.full(times_utc.shape, np.nan, dtype=np.float32), source
    north = np.interp(times_utc, pos_t, ned[:, 0])
    east = np.interp(times_utc, pos_t, ned[:, 1])
    down = np.interp(times_utc, pos_t, ned[:, 2])
    distance = np.sqrt(north**2 + east**2 + down**2)
    return distance.astype(np.float32), source

## Build Labeled Audio Windows

This cell creates one row per training window. It stores the audio samples in memory so feature extraction can run without rereading each `.mat` file.

In [5]:
@dataclass(frozen=True)
class WindowExample:
    file: str
    vehicle: str
    mode: str
    flight: int
    start_s: float
    end_s: float
    distance_m: float
    label: int
    label_name: str
    position_source: str
    samples: np.ndarray


def label_window(vehicle: str, distance_m: float) -> tuple[int, str] | None:
    if vehicle in MULTIROTOR_VEHICLES:
        if np.isfinite(distance_m) and distance_m <= POSITIVE_DISTANCE_M:
            return 1, 'drone'
        if np.isfinite(distance_m) and distance_m >= BACKGROUND_DISTANCE_M:
            return 0, 'no_drone'
        return None
    if INCLUDE_FIXED_WING_AS_NEGATIVE and vehicle in FIXED_WING_VEHICLES:
        return 0, 'no_drone'
    return None


def build_windows_for_file(row: pd.Series) -> list[WindowExample]:
    audio, utc_time, sr = get_audio_and_time(row.path, row.vehicle)
    window_n = int(round(WINDOW_S * sr))
    hop_n = int(round(HOP_S * sr))
    starts = np.arange(0, max(1, len(audio) - window_n + 1), hop_n, dtype=int)
    if MAX_WINDOWS_PER_FILE is not None and len(starts) > MAX_WINDOWS_PER_FILE:
        rng = np.random.default_rng(RANDOM_SEED + int(row.flight))
        starts = np.sort(rng.choice(starts, size=MAX_WINDOWS_PER_FILE, replace=False))

    examples: list[WindowExample] = []
    for start in starts:
        end = start + window_n
        if end > len(audio):
            continue
        mid = start + window_n // 2
        mid_utc = utc_time[min(mid, len(utc_time) - 1)]
        distance, source = distance_at_times(row.path, np.asarray([mid_utc]))
        distance_m = float(distance[0])
        label = label_window(row.vehicle, distance_m)
        if label is None:
            continue
        y, label_name = label
        examples.append(
            WindowExample(
                file=row.file,
                vehicle=row.vehicle,
                mode=row.mode,
                flight=int(row.flight),
                start_s=float(start / sr),
                end_s=float(end / sr),
                distance_m=distance_m,
                label=y,
                label_name=label_name,
                position_source=source,
                samples=audio[start:end].copy(),
            )
        )
    return examples

examples: list[WindowExample] = []
for row in flight_df.itertuples(index=False):
    examples.extend(build_windows_for_file(pd.Series(row._asdict())))

meta = pd.DataFrame(
    {
        'file': ex.file,
        'vehicle': ex.vehicle,
        'mode': ex.mode,
        'flight': ex.flight,
        'start_s': ex.start_s,
        'end_s': ex.end_s,
        'distance_m': ex.distance_m,
        'label': ex.label,
        'label_name': ex.label_name,
        'position_source': ex.position_source,
    }
    for ex in examples
)
print('Windows:', len(examples))
display(meta.groupby(['vehicle', 'mode', 'label_name']).size().rename('windows').reset_index())
display(meta['label_name'].value_counts())

Windows: 1877


TypeError: '<' not supported between instances of 'method' and 'method'

## Extract Features

The feature extractor is the same baseline MFCC feature function used by the app-compatible audio module. That keeps the exported model compatible with the dashboard fallback loader.

In [ ]:
if not examples:
    raise RuntimeError('No labeled windows were built. Loosen the distance thresholds or inspect GPS data.')

X = np.vstack([extract_baseline_features_from_signal(ex.samples, TARGET_SR) for ex in examples])
y = np.asarray([ex.label for ex in examples], dtype=np.int64)
groups = np.asarray([ex.file for ex in examples])
print('X:', X.shape, 'y:', y.shape, 'positive rate:', y.mean().round(3))

## Train/Test Split By Flight

The split groups by file name so windows from the same flight are not shared between training and testing.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

clf = Pipeline(
    steps=[
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)),
    ]
)
clf.fit(X[train_idx], y[train_idx])

pred = clf.predict(X[test_idx])
proba = clf.predict_proba(X[test_idx])[:, 1]
print('Train windows:', len(train_idx), 'Test windows:', len(test_idx))
print('Train files:', len(set(groups[train_idx])), 'Test files:', len(set(groups[test_idx])))
print(confusion_matrix(y[test_idx], pred))
print(classification_report(y[test_idx], pred, target_names=['no_drone', 'drone']))

## Inspect Predictions

Use this to find hard examples and decide whether the thresholds need adjustment.

In [ ]:
results = meta.iloc[test_idx].copy().reset_index(drop=True)
results['score'] = proba
results['pred'] = pred
results['pred_name'] = np.where(results['pred'] == 1, 'drone', 'no_drone')
results['correct'] = results['pred'] == results['label']

display(results.sort_values('score', ascending=False).head(12))
display(results.sort_values('score', ascending=True).head(12))
display(results.groupby(['vehicle', 'label_name', 'pred_name']).size().rename('windows').reset_index())

## Export Model

This writes a model to the path used by the dashboard fallback audio backend:

```text
Project v1/src/audio/models/sound_baseline_mfcc_logreg.joblib
```

If you do not want to replace the current fallback model, change `MODEL_OUT` above before running this cell.

In [ ]:
artifact = {
    'model': clf,
    'target_sr': TARGET_SR,
    'window_s': WINDOW_S,
    'hop_s': HOP_S,
    'label_strategy': {
        'positive_vehicles': sorted(MULTIROTOR_VEHICLES),
        'negative_vehicles': sorted(FIXED_WING_VEHICLES) if INCLUDE_FIXED_WING_AS_NEGATIVE else [],
        'positive_distance_m': POSITIVE_DISTANCE_M,
        'background_distance_m': BACKGROUND_DISTANCE_M,
    },
}

# The app's baseline loader expects a direct sklearn estimator/pipeline with predict_proba.
# Save the pipeline itself for compatibility.
joblib.dump(clf, MODEL_OUT)
print('Saved app-compatible model:', MODEL_OUT)

metadata_out = MODEL_OUT.with_suffix('.metadata.json')
metadata_out.write_text(json.dumps({k: v for k, v in artifact.items() if k != 'model'}, indent=2), encoding='utf-8')
print('Saved metadata:', metadata_out)

## Optional: Export A Scenario WAV From A `.mat` File

The dashboard expects a normal audio file such as `scenario-1.wav`. This cell converts one NASA `.mat` recording into `.wav` using the same microphone selection as the training code.

In [ ]:
EXAMPLE_MAT = MAT_DIR / 'phantom_flyover_120.mat'
SCENARIO_DIR = Path('/home/mo/dev/python/HDDS2-datasets/generated_scenarios')
SCENARIO_DIR.mkdir(parents=True, exist_ok=True)
OUT_WAV = SCENARIO_DIR / 'phantom_flyover_120.wav'

info = parse_mat_name(EXAMPLE_MAT)
audio, _utc_time, sr = get_audio_and_time(EXAMPLE_MAT, info['vehicle'])
sf.write(OUT_WAV, audio, int(sr))
print('Wrote:', OUT_WAV)
print('Duration seconds:', round(len(audio) / sr, 2), 'Sample rate:', sr)

## Optional: Create A Matching Radar JSON Stub

This is not measured radar. It is a simulation parameter file derived from the GPS track so the dashboard can run a synchronized scenario. Pair it with your own video file by using the same stem.

In [ ]:
pos_t, ned, source = get_position_track(EXAMPLE_MAT)
if pos_t is None or ned is None:
    raise RuntimeError('No GPS/RTK position track found for this file.')

distance = np.sqrt(np.sum(ned**2, axis=1))
if len(pos_t) > 1:
    speed = np.sqrt(np.sum(np.diff(ned, axis=0)**2, axis=1)) / np.maximum(np.diff(pos_t), 1e-6)
    representative_speed = float(np.nanmedian(speed))
else:
    representative_speed = 10.0

radar_json = {
    'name': EXAMPLE_MAT.stem,
    'noise_power_db': -75,
    'clutter_amplitude_db': -45,
    'targets': [
        {
            'name': info['vehicle'],
            'distance_m': float(np.nanmedian(distance)),
            'speed_mps': representative_speed,
            'amplitude_db': -18,
        }
    ],
}
OUT_JSON = SCENARIO_DIR / f'{EXAMPLE_MAT.stem}.json'
OUT_JSON.write_text(json.dumps(radar_json, indent=2), encoding='utf-8')
print('Wrote:', OUT_JSON)
print(json.dumps(radar_json, indent=2))

## Notes

This notebook fixes the model-training side, but the quality depends on labels. The NASA dataset does not include explicit `drone/no-drone` labels per second, so this notebook uses vehicle type and GPS distance as practical proxy labels.

Good next improvements:
- add real background/no-drone recordings,
- tune distance thresholds after listening to examples,
- compare MFCC logistic regression against a YAMNet embedding model,
- evaluate on scenario audio that was not used during training.